In [1]:
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.embeddings import DashScopeEmbeddings
from ragas.testset.persona import Persona
from ragas.testset.transforms.extractors.llm_based import NERExtractor
from ragas.testset.transforms.splitters import HeadlineSplitter
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers.single_hop.specific import (
    SingleHopSpecificQuerySynthesizer,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

In [4]:
os.environ["OPENAI_API_KEY"] = "sk-4rVuUWLOC3GHtKq8UQM0vmfMwZGYMYe4Kj5lIOljp3SnkMXo"
# export OPENAPI_HOST="https://openapi.test.dp.tech" 
os.environ["OPENAI_API_BASE"] = "https://openai.weavex.tech/v1/"
os.environ["DASHSCOPE_API_KEY"] = "sk-f22c4fa77bab4a42a47486922c84a467"
path = "/Users/xiaohuxu/Documents/python/sob_blobs/sobereva_blogs_text/about_qc/test"
loader = DirectoryLoader(path, glob="*.md")
docs = loader.load()
print(len(docs))

chatmodel =  ChatOpenAI(
                model="gpt-4o-mini",
                base_url="https://openai.weavex.tech/v1"
                )

chatmodel = ChatOpenAI(
    api_key="sk-f22c4fa77bab4a42a47486922c84a467",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model="qwen-plus",
)

# embeddingmodel = OpenAIEmbeddings(
#     model="text-embedding-3-small",
#     openai_api_base="https://openai.weavex.tech/v1/"
#     )
embeddingmodel = DashScopeEmbeddings(model="text-embedding-v4",)

generator_llm = LangchainLLMWrapper(chatmodel)
generator_embeddings = LangchainEmbeddingsWrapper(embeddingmodel)

personas = [
    Persona(
        name="充满好奇心的学生",
        role_description="你是一个对计算化学和量子化学充满好奇心的学生，渴望了解更多关于这些领域的知识。",
    ),
]

# transforms = [HeadlineSplitter()]
transforms = [HeadlineSplitter(), NERExtractor()]

generator = TestsetGenerator(
    llm=generator_llm, 
    embedding_model=generator_embeddings, 
    persona_list=personas
)

distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 1.0),
]


# for query, _ in distribution:
#     prompts = await query.adapt_prompts("chinese", llm=generator_llm)
#     query.set_prompts(**prompts)

dataset = generator.generate_with_langchain_docs(
    docs[:],
    testset_size=5,
    transforms=transforms,
    query_distribution=distribution,
)

eval_dataset = dataset.to_evaluation_dataset()

print("Query:", eval_dataset[0].user_input)
print("Reference:", eval_dataset[0].reference)


4


Applying HeadlineSplitter:   0%|          | 0/4 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying NERExtractor:   0%|          | 0/4 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/6 [00:00<?, ?it/s]

Query: 在计算化学领域，Core 2 Q6600硬件在Amber14的安装和测试过程中扮演了什么角色？
Reference: 在Amber14的安装和测试过程中，Core 2 Q6600作为硬件平台被用于编译和运行Amber14及其相关模块。安装过程中明确提到了使用该硬件进行测试，包括串行版本、并行版本以及GPU加速版本的编译和测试。在并行测试中，使用了命令"mpirun -np 4"来运行四核测试，表明Core 2 Q6600是一个四核处理器。


In [6]:
eval_dataset.to_pandas()

,user_input,reference_contexts,reference
0,在计算化学领域，Core 2 Q6600硬件在Amber14的安装和测试过程中扮演了什么角色？,[Amber14安装方法\n\nURL: http://sobereva.com/263 I...,在Amber14的安装和测试过程中，Core 2 Q6600作为硬件平台被用于编译和运行Am...
1,AmberTools1.5 和 Amber14 的关系是什么？,[Amber14安装方法\n\nURL: http://sobereva.com/263 I...,AmberTools1.5 与 Amber14 密切相关，因为 Amber 的许多功能已经被...
2,ZORA在相对论量子化学中指的是什么？,[使用Gaussian+PySOC在TDDFT下计算旋轨耦合矩阵元\n\nURL: http...,ZORA（Zero-order regular approximation）是一种将Dira...
3,作为一个对计算化学和量子化学充满好奇心的学生，我想了解在使用Gaussian结合PySOC程...,[使用Gaussian+PySOC在TDDFT下计算旋轨耦合矩阵元\n\nURL: http...,使用Gaussian结合PySOC程序可以在TDDFT级别下计算单重态与三重态之间的旋轨耦合...
4,Multiwfn在ORCA量子化学程序的使用中扮演什么角色，并且如何帮助用户更高效地进行计算...,[量子化学程序ORCA的安装方法\n\nURL: http://sobereva.com/4...,Multiwfn可以用于生成ORCA量子化学程序的输入文件，里面的关键词设置绝对恰当，简化了...
5,cmder在使用ORCA时有什么作用？,[量子化学程序ORCA的安装方法\n\nURL: http://sobereva.com/4...,利用cmder可以令ORCA在Windows下的使用明显更方便。cmder是一个第三方的文本...


In [11]:
def print_item(idx):
    print("Query:", eval_dataset[idx].user_input)
    print("Reference:", eval_dataset[idx].reference)
    # print("Context:", eval_dataset[idx].reference_contexts)
print_item(3)

Query: 作为一个对计算化学和量子化学充满好奇心的学生，我想了解在使用Gaussian结合PySOC程序进行TDDFT计算时，如何获得单重态与三重态之间的旋轨耦合矩阵元，并且这些矩阵元在研究磷光发射和系间窜越速率方面有哪些具体应用？
Reference: 使用Gaussian结合PySOC程序可以在TDDFT级别下计算单重态与三重态之间的旋轨耦合矩阵元。首先，用户需用Gaussian09运行TD(50-50,nstates=x)任务，计算单重态和三重态各x个，并保留rwf文件。PySOC会调用Gaussian的rwfdump工具从rwf文件中提取组态系数、分子轨道展开系数等信息，并结合输出文件中的激发能和基组定义，调用MolSOC程序以Zeff方式计算旋轨耦合积分，最终通过组合这些数据得到旋轨耦合矩阵元。旋轨耦合矩阵元的单位通常为cm^-1。这些矩阵元在研究磷光发射速率和系间窜越速率方面有重要应用：通过微扰方式考虑旋轨耦合效应，可以计算磷光发射速率；而系间窜越速率正比于相应两个态之间旋轨耦合矩阵元的模方。此外，旋轨耦合矩阵元还可用于计算多重态子态的能级分裂（零场分裂）。
